In [1]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
import xgboost as xgb
from xgboost import XGBClassifier
#from lightgbm import LGBMClassifier
#from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn import metrics
import seaborn as sns
import numpy as np
import warnings

In [2]:
target_column = "health_condition"

In [3]:
df = pd.read_csv("data/train.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  str    
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  str    
 10  stress_level             607277 non-null  str    
 11  sleep_quality            631757 non-null  str    
 12  physical_activity_level  653467 non-null  str    
 13  smoking_alcohol          661506 non-null  str    
 14  gender         

In [4]:
df['diet_type'] = df['diet_type'].astype('category')
df['stress_level'] = df['stress_level'].replace({'low':0, 'medium':1, 'high':2})
df['sleep_quality'] = df['sleep_quality'].replace({'poor':0, 'average':1, 'good':2})
df['physical_activity_level'] = df['physical_activity_level'].replace({'sedentary':0, 'moderate':1, 'active':2})
df['smoking_alcohol'] = df['smoking_alcohol'].replace({'no':0, 'occasional':1, 'yes':2})
df['gender'] = df['gender'].astype('category')

object_cols = ['stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol']
for col in object_cols:
    df[col] = df[col].astype('category')
# convert object types to category
# TODO - make this ordered again somehow

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   id                       690088 non-null  int64   
 1   health_condition         690088 non-null  str     
 2   sleep_duration           614089 non-null  float64 
 3   heart_rate               682255 non-null  float64 
 4   bmi                      676190 non-null  float64 
 5   calorie_expenditure      637235 non-null  float64 
 6   step_count               676172 non-null  float64 
 7   exercise_duration        683187 non-null  float64 
 8   water_intake             646611 non-null  float64 
 9   diet_type                683187 non-null  category
 10  stress_level             607277 non-null  category
 11  sleep_quality            631757 non-null  category
 12  physical_activity_level  653467 non-null  category
 13  smoking_alcohol          661506 non-null  category
 14 

In [5]:
df['calorie_expenditure_per_step'] = df['calorie_expenditure'] / (df['step_count']+1)
df['step_speed'] = df['step_count'] / (df['exercise_duration']+1)
df['calorie_expenditure_per_min'] = df['calorie_expenditure'] / (df['exercise_duration']+1)
df['calorie_expenditure_per_bmi'] = df['calorie_expenditure'] / (df['bmi']+1)

In [6]:
df.drop('id', axis=1)

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,calorie_expenditure_per_step,step_speed,calorie_expenditure_per_min,calorie_expenditure_per_bmi
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,2,1,0,2,female,1.638282,63.750000,104.519231,81.545386
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,0,1,1,2,other,0.198746,194.322200,38.624754,73.248882
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,2,0,2,2,male,0.189069,363.580563,68.746803,105.246672
3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,2,1,2,1,female,0.366551,117.799672,43.185550,108.992955
4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,1,0,NaN,male,0.388762,140.085106,54.468085,86.956522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690083,at-risk,6.31,69.7,24.11,2157.0,NaN,30.8,3.00,non-veg,2,0,2,2,female,NaN,NaN,67.830189,85.902031
690084,at-risk,5.78,54.0,26.36,2858.0,6488.0,52.4,1.46,veg,1,1,1,0,male,0.440438,121.498127,53.520599,104.459064
690085,fit,7.64,85.7,21.91,2195.0,9241.0,41.3,1.57,non-veg,NaN,1,2,0,male,0.237503,218.463357,51.891253,95.809690
690086,at-risk,6.74,73.0,18.73,2061.0,13366.0,56.6,2.60,balanced,NaN,1,2,2,male,0.154186,232.048611,35.781250,104.460213


In [7]:
y = df[target_column].replace({'unhealthy':0, 'at-risk':1, 'fit':2}).astype('int')
X = df.drop(target_column, axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
df_test = pd.read_csv("data/test.csv")

df_submission = pd.read_csv("data/sample_submission.csv")
df_submission.info()

<class 'pandas.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                295753 non-null  int64
 1   health_condition  295753 non-null  str  
dtypes: int64(1), str(1)
memory usage: 4.5 MB


In [9]:
df_test['diet_type'] = df_test['diet_type'].astype('category')
df_test['stress_level'] = df_test['stress_level'].replace({'low':0, 'medium':1, 'high':2})
df_test['sleep_quality'] = df_test['sleep_quality'].replace({'poor':0, 'average':1, 'good':2})
df_test['physical_activity_level'] = df_test['physical_activity_level'].replace({'sedentary':0, 'moderate':1, 'active':2})
df_test['smoking_alcohol'] = df_test['smoking_alcohol'].replace({'no':0, 'occasional':1, 'yes':2})
df_test['gender'] = df_test['gender'].astype('category')

object_cols = ['stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol']
for col in object_cols:
    df_test[col] = df_test[col].astype('category')

df_test['calorie_expenditure_per_step'] = df_test['calorie_expenditure'] / (df_test['step_count']+1)
df_test['step_speed'] = df_test['step_count'] / (df_test['exercise_duration']+1)
df_test['calorie_expenditure_per_min'] = df_test['calorie_expenditure'] / (df_test['exercise_duration']+1)
df_test['calorie_expenditure_per_bmi'] = df_test['calorie_expenditure'] / (df_test['bmi']+1)

In [10]:
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

model = joblib.load('models/xgboost_tts_80.pkl')
model.fit(X_train, y_train, sample_weight=train_sample_weight, eval_set=[(X_test, y_test)], verbose=0)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,'gbtree'
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",100
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegre

In [11]:
test_sample_weight = compute_sample_weight(class_weight="balanced", y=y_test)
val_score = model.score(X_test, y_test, sample_weight=test_sample_weight)
val_score

0.9504715152293938

In [12]:
X_train.columns
test_features = ['id', 'diet_type', 'calorie_expenditure_per_bmi', 'calorie_expenditure_per_step', 'step_speed', 'calorie_expenditure', 'calorie_expenditure_per_min', 'gender']
rest_of_features = list(set(X_train.columns).difference(set(test_features)))
remove_features = (test_features + rest_of_features)
remove_features.pop()
remove_features

['id',
 'diet_type',
 'calorie_expenditure_per_bmi',
 'calorie_expenditure_per_step',
 'step_speed',
 'calorie_expenditure',
 'calorie_expenditure_per_min',
 'gender',
 'sleep_quality',
 'step_count',
 'heart_rate',
 'physical_activity_level',
 'smoking_alcohol',
 'bmi',
 'water_intake',
 'stress_level',
 'exercise_duration']

In [13]:
val_scores = []

for feature in remove_features:
    X.drop(feature, axis=1, inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = joblib.load('models/xgboost_tts_80.pkl')
    model.fit(X_train, y_train, sample_weight=train_sample_weight, eval_set=[(X_test, y_test)], verbose=0)

    feature_val_score = model.score(X_test, y_test, sample_weight=test_sample_weight)
    val_scores.append(feature_val_score)

score_df = pd.DataFrame({'Feature': pd.Series(remove_features), 'Validation Score': pd.Series(val_scores)})
score_df

,Feature,Validation Score
0,id,0.950384
1,diet_type,0.950343
2,calorie_expenditure_per_bmi,0.950181
3,calorie_expenditure_per_step,0.950289
4,step_speed,0.950210
5,calorie_expenditure,0.950452
6,calorie_expenditure_per_min,0.950444
7,gender,0.950326
8,sleep_quality,0.949808
9,step_count,0.949487
